# Development notebook — separate measurement protocol
This working copy contains earlier local benchmark repairs. It is not the frozen four-path experiment underlying the current headline table.
Use [the frozen four-path notebook](../benchmarks/notebooks/all-paths-speed.ipynb) for that protocol.
See [tool status](../docs/BENCHMARK_TOOL_STATUS.md) for the distinction from the previously published historical wrapper.


# Atomic GDN2: equal-work benchmark (pending review)

This notebook does not assume any historical speedup. It runs on one TPU v5e device.
Sources are pinned to the original kernel revision; the revised harness is embedded below.
Results are private to your session/Drive until you decide to publish.

## Protocol

- Identical FP32 inputs, nonzero initial state, output scaling and final-state objective.
- A small token-serial oracle checks outputs and all seven input gradients first.
- Every measured seed/shape must pass full-size output, final-state and gradient checks.
- Correct associative scan and ordinary JAX WY are both reported as baselines.
- Strict profile: zero damping, highest matmul precision. Production profile: original
  1e-3 damping and bf16x3 solve/B3; discrepancies are measured against the exact recurrence.
- Fixed relative-L2 tolerance 0.005, selected before execution. No timing if it fails.
- Compile and first execution excluded; three further warm-ups, 15 synchronized timed
  repetitions per seed, randomized interleaving, seeds 0/1/2. Every sample is saved.
- Full E2E uses a new 27.33M tied-embedding five-layer LM, cross-entropy and Adam;
  B8/L2048 and identical microbatch 2 for both paths. Ten repetitions after warm-up.
  Inputs are synthetic random token IDs; corpus loading/checkpoint I/O are excluded.
  It replays the same initial optimizer state; this is throughput, not a training-quality run.
- E2E is a new specified workload, not a reconstruction of the undocumented 200.17x run.
- No GitHub/HF push or replacement of old results occurs here.


In [ ]:
import pathlib, subprocess, sys, urllib.request, zipfile
subprocess.check_call([sys.executable,'-m','pip','install','-q','jax[tpu]==0.11.2','jaxlib==0.11.2','libtpu==0.0.48.1','numpy==2.1.3'])
url='https://github.com/Akseleu-J/atomic-ops/archive/11616acd8a061768a38feb12c58e5e329952f5d5.zip'
urllib.request.urlretrieve(url,'/content/atomic-source.zip')
with zipfile.ZipFile('/content/atomic-source.zip') as z:z.extractall('/content/atomic-source')
source_dir=next(pathlib.Path('/content/atomic-source').iterdir())
print('Kernel source:',source_dir)


## honest_gdn2.py

In [ ]:
(source_dir/'benchmarks'/'honest_gdn2.py').write_text('"""Equal-work GDN2 benchmark; every published ratio requires numerical gates.\n\nNo historical target speedup is assumed. TPU measurements are not CPU interpretation.\nAll wrappers use BLHD inputs, FP32 storage, common nonzero h0 and output scale.\n"""\nfrom __future__ import annotations\nimport argparse, dataclasses, hashlib, importlib.metadata, json, math, os, platform\nimport statistics, subprocess, sys, time, traceback\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport jax\nimport jax.numpy as jnp\nimport numpy as np\n\nROOT=Path(__file__).resolve().parents[1]\nsys.path.insert(0,str(ROOT))\nfrom atomic_gdn2 import make_cfg, make_blr_trainable\nfrom atomic_ops.reference import gdn2_token_serial_reference, gdn2_chunked_wy_reference\n\nPREC=jax.lax.Precision.HIGHEST\nARGNAMES=(\'q\',\'k\',\'v\',\'w\',\'b\',\'g\',\'h0\')\nTOL=.005  # fixed before execution; never relaxed after seeing a result\n\ndef associative(q,k,v,w,b,g,h0,scale):\n    """Pure affine composition; form matrices inside each chunk, not for all L."""\n    B,L,H,D=q.shape; C=128; N=L//C\n    def chunk(x):return jnp.moveaxis(x.reshape(B,N,C,H,D),1,0)\n    def combine(left,right):\n        ml,cl=left;mr,cr=right\n        return jnp.matmul(mr,ml,precision=PREC),jnp.matmul(mr,cl,precision=PREC)+cr\n    def step(h,xs):\n        qq,kk,vv,ww,bb,gg=xs\n        alpha=jnp.exp(gg)\n        m=jnp.eye(D)*alpha[...,None,:]-kk[..., :,None]*((bb*kk)*alpha)[...,None,:]\n        c=kk[..., :,None]*(ww*vv)[...,None,:]\n        pm,pc=jax.lax.associative_scan(combine,(m,c),axis=1)\n        states=jnp.matmul(pm,h[:,None],precision=PREC)+pc\n        out=scale*jnp.einsum(\'bchdv,bchd->bchv\',states,qq,precision=PREC)\n        return states[:,-1],out\n    hf,out=jax.lax.scan(jax.checkpoint(step),h0,tuple(map(chunk,(q,k,v,w,b,g))))\n    return jnp.moveaxis(out,0,1).reshape(B,L,H,D),hf\n\ndef make_paths(D,profile):\n    scale=1/math.sqrt(D)\n    strict=profile==\'strict\'\n    cfg=make_cfg(bt=128,bc=64,score_bs=128,bs2=64,mb=8,wy_eps=0.0 if strict else .001,\n        dot_mode=\'highest\',solve_dot_mode=\'highest\' if strict else \'bf16x3\',\n        b3_dot_mode=\'highest\' if strict else \'bf16x3\',interpret=False)\n    kernel=make_blr_trainable(cfg,scale)\n    def pallas(q,k,v,w,b,g,h0):\n        o,h=kernel(*(jnp.swapaxes(x,1,2) for x in (q,k,v,w,b,g)),h0)\n        return jnp.swapaxes(o,1,2),h\n    def scan(*xs):return associative(*xs,scale)\n    def serial(q,k,v,w,b,g,h0):return gdn2_token_serial_reference(q,k,v,g,b,w,scale,h0)\n    def wy(q,k,v,w,b,g,h0):\n        return gdn2_chunked_wy_reference(q,k,v,g,b,w,scale,128,h0,wy_eps=0.0)\n    return dict(associative_scan=scan,jax_wy=wy,pallas=pallas,serial=serial),dataclasses.asdict(cfg)\n\ndef inputs(shape,seed):\n    B,L,H,D=shape;r=np.random.default_rng(seed)\n    mk=lambda scale:r.normal(size=shape).astype(np.float32)*scale\n    q,k,v=mk(.3),mk(.3),mk(.3)\n    q=q/np.sqrt(np.sum(q*q,axis=-1,keepdims=True)+1e-6)\n    k=k/np.sqrt(np.sum(k*k,axis=-1,keepdims=True)+1e-6)\n    w=r.uniform(.2,1,shape).astype(np.float32)\n    b=r.uniform(.2,1,shape).astype(np.float32)\n    g=-np.abs(mk(.05))\n    h0=r.normal(size=(B,H,D,D)).astype(np.float32)*.1\n    arrays=(q,k,v,w,b,g,h0)\n    digest=hashlib.sha256(b\'\'.join(x.tobytes() for x in arrays)).hexdigest()\n    return tuple(jax.device_put(x) for x in arrays),digest\n\ndef loss_for(f):\n    def loss(*xs):\n        o,h=f(*xs)\n        return jnp.sum(o*o)+jnp.sum(h*h)\n    return loss\n\ndef compare(a,b):\n    aa=np.asarray(a,np.float64);bb=np.asarray(b,np.float64)\n    finite=bool(np.isfinite(aa).all() and np.isfinite(bb).all())\n    if not finite:return dict(finite=False,rel_l2=None,max_abs=None,reference_norm=None)\n    diff=aa-bb\n    return dict(finite=finite,rel_l2=float(np.linalg.norm(diff)/(np.linalg.norm(bb)+1e-12)),\n                max_abs=float(np.max(np.abs(diff))),reference_norm=float(np.linalg.norm(bb)))\n\ndef tree_comparison(a,b,names):\n    result={name:compare(x,y) for name,x,y in zip(names,jax.tree.leaves(a),jax.tree.leaves(b))}\n    passed=all(v[\'finite\'] and v[\'rel_l2\']<=TOL for v in result.values())\n    return dict(passed=passed,tensors=result)\n\ndef compile_fn(f,args):\n    start=time.perf_counter(); exe=jax.jit(f).lower(*args).compile()\n    duration=time.perf_counter()-start\n    jax.block_until_ready(exe(*args))\n    return exe,duration\n\ndef timings(exes,args,repeats,warmup,seed):\n    for f in exes.values():\n        for _ in range(warmup):jax.block_until_ready(f(*args))\n    raw={name:[] for name in exes};order=[];rng=np.random.default_rng(seed)\n    jax.block_until_ready(args)\n    for i in range(repeats):\n        names=list(exes);rng.shuffle(names);order.append(names)\n        for name in names:\n            begin=time.perf_counter();out=exes[name](*args);jax.block_until_ready(out)\n            raw[name].append((time.perf_counter()-begin)*1000)\n    return dict(samples={name:dict(samples_ms=s,median_ms=statistics.median(s),\n        mean_ms=statistics.mean(s),sample_sd_ms=statistics.stdev(s) if len(s)>1 else 0.,\n        min_ms=min(s),max_ms=max(s)) for name,s in raw.items()},order=order)\n\ndef environment():\n    versions={}\n    for p in (\'jax\',\'jaxlib\',\'libtpu\',\'numpy\'):\n        try:versions[p]=importlib.metadata.version(p)\n        except importlib.metadata.PackageNotFoundError:versions[p]=None\n    return dict(utc=datetime.now(timezone.utc).isoformat(),python=sys.version,platform=platform.platform(),\n        packages=versions,backend=jax.default_backend(),devices=[str(x) for x in jax.devices()],\n        device_kinds=[x.device_kind for x in jax.devices()],local_devices=jax.local_device_count(),\n        matmul_precision=str(jax.config.jax_default_matmul_precision),\n        environment={k:os.environ.get(k) for k in (\'XLA_FLAGS\',\'JAX_PLATFORMS\',\'TPU_NAME\')})\n\ndef run_micro(a):\n    shape=tuple(map(int,a.shape.split(\',\')));D=shape[-1]\n    assert len(shape)==4 and shape[1]%128==0\n    paths,cfg=make_paths(D,a.profile)\n    report=dict(kind=\'kernel\',shape=shape,profile=a.profile,config=cfg,\n        objective=\'sum(o**2)+sum(h_final**2)\',argnums=list(ARGNAMES),storage_dtype=\'float32\',\n        output_scale=1/math.sqrt(D),initial_state=\'nonzero, identical for every path\',\n        tolerance_rel_l2=TOL,environment=environment(),repeats=a.repeats,warmup=a.warmup,\n        compile_excluded=True,transfer_excluded=True,seeds=[],ratios=[])\n    small,_=inputs((1,128,1,D),12345)\n    serial,serial_sec=compile_fn(paths[\'serial\'],small)\n    serial_grad,_=compile_fn(jax.value_and_grad(loss_for(paths[\'serial\']),argnums=tuple(range(7))),small)\n    oracle=serial(*small);oracle_grad=serial_grad(*small)\n    report[\'small_oracle\']={}\n    for name in (\'associative_scan\',\'jax_wy\',\'pallas\'):\n        f,_=compile_fn(paths[name],small)\n        gf,_=compile_fn(jax.value_and_grad(loss_for(paths[name]),argnums=tuple(range(7))),small)\n        forward=tree_comparison(f(*small),oracle,(\'o\',\'h_final\'))\n        grad=tree_comparison(gf(*small),oracle_grad,(\'loss\',)+ARGNAMES)\n        report[\'small_oracle\'][name]=dict(forward=forward,gradient=grad)\n    if not all(r[\'forward\'][\'passed\'] and r[\'gradient\'][\'passed\'] for r in report[\'small_oracle\'].values()):\n        report[\'status\']=\'oracle_gate_failed\';return report\n    x,_=inputs(shape,a.seeds[0]);exes={};compile_s={};fwds={}\n    for name in (\'associative_scan\',\'jax_wy\',\'pallas\'):\n        print(\'COMPILE\',name,shape,a.profile,flush=True)\n        fwds[name],_=compile_fn(paths[name],x)\n        exes[name],compile_s[name]=compile_fn(jax.value_and_grad(loss_for(paths[name]),argnums=tuple(range(7))),x)\n    report[\'compile_seconds\']=compile_s\n    for seed in a.seeds:\n        xs,digest=inputs(shape,seed)\n        ref=fwds[\'associative_scan\'](*xs);ref_grad=exes[\'associative_scan\'](*xs)\n        checks={}\n        for name in (\'jax_wy\',\'pallas\'):\n            checks[name]=dict(forward=tree_comparison(fwds[name](*xs),ref,(\'o\',\'h_final\')),\n                gradient=tree_comparison(exes[name](*xs),ref_grad,(\'loss\',)+ARGNAMES))\n        row=dict(seed=seed,input_sha256=digest,checks=checks)\n        row[\'passed\']=all(v[\'forward\'][\'passed\'] and v[\'gradient\'][\'passed\'] for v in checks.values())\n        if row[\'passed\']:\n            row[\'timing\']=timings(exes,xs,a.repeats,a.warmup,seed)\n            times=row[\'timing\'][\'samples\'];row[\'speedups\']={\n                name:times[name][\'median_ms\']/times[\'pallas\'][\'median_ms\'] for name in (\'associative_scan\',\'jax_wy\')}\n        report[\'seeds\'].append(row)\n        print(\'SEED_RESULT\',json.dumps(row),flush=True)\n    report[\'status\']=\'passed\' if all(r[\'passed\'] for r in report[\'seeds\']) else \'gate_failed\'\n    if report[\'status\']==\'passed\':\n        report[\'speedups\']={name:dict(per_seed=[r[\'speedups\'][name] for r in report[\'seeds\']],\n            mean=statistics.mean(r[\'speedups\'][name] for r in report[\'seeds\']),\n            sample_sd=statistics.stdev(r[\'speedups\'][name] for r in report[\'seeds\']) if len(a.seeds)>1 else 0.)\n            for name in (\'associative_scan\',\'jax_wy\')}\n    return report\n\ndef main():\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--shape\',default=\'4,2048,6,128\');p.add_argument(\'--profile\',choices=(\'strict\',\'production\'),default=\'strict\')\n    p.add_argument(\'--seeds\',type=int,nargs=\'+\',default=[0,1,2]);p.add_argument(\'--repeats\',type=int,default=15)\n    p.add_argument(\'--warmup\',type=int,default=3);p.add_argument(\'--output\',default=\'honest-benchmark.json\')\n    p.add_argument(\'--allow-cpu-oracle\',action=\'store_true\')\n    a=p.parse_args()\n    assert a.repeats>=3 and a.warmup>=1\n    jax.config.update(\'jax_default_matmul_precision\',\'highest\')\n    if a.allow_cpu_oracle:\n        paths,_=make_paths(8,\'strict\');xs,_=inputs((1,128,1,8),0)\n        # CPU checks only pure JAX wrappers, never report Pallas interpretation speed.\n        for key in (\'associative_scan\',\'jax_wy\'):\n            check=tree_comparison(jax.jit(paths[key])(*xs),jax.jit(paths[\'serial\'])(*xs),(\'o\',\'hf\'))\n            assert check[\'passed\'],check\n        print(\'CPU reference oracle PASS; no TPU speed measured\');return\n    assert jax.default_backend()==\'tpu\' and jax.local_device_count()==1,\'Requires exactly one TPU device\'\n    try:report=run_micro(a)\n    except Exception as error:\n        report=dict(status=\'error\',error=type(error).__name__,detail=str(error),traceback=traceback.format_exc(),environment=environment())\n    report[\'benchmark_sha256\']=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n    report[\'kernel_source_sha256\']={str(f.relative_to(ROOT)):hashlib.sha256(f.read_bytes()).hexdigest()\n        for f in (ROOT/\'atomic_gdn2\').rglob(\'*.py\')}\n    report[\'source_revision\']=\'11616acd8a061768a38feb12c58e5e329952f5d5\'\n    Path(a.output).write_text(json.dumps(report,indent=2,allow_nan=False))\n    print(\'FINAL\',json.dumps(report.get(\'speedups\',{})),report[\'status\'],flush=True)\n    return 0 if report[\'status\']==\'passed\' else 1\n\nif __name__==\'__main__\':raise SystemExit(main())\n',encoding='utf-8')


## honest_e2e.py

In [ ]:
(source_dir/'benchmarks'/'honest_e2e.py').write_text('"""Same-model same-Adam-update benchmark. This is a new protocol, not a recovered old log."""\nimport argparse, hashlib, json, math, os, statistics, sys, time, traceback\nfrom pathlib import Path\nsys.path.insert(0,str(Path(__file__).resolve().parent))\nfrom honest_gdn2 import jax,jnp,np,PREC,TOL,make_paths,tree_comparison,compile_fn,environment\n\ndef init(seed,width=384,ff=768,layers=5,vocab=32768,heads=6,head_dim=128):\n    rng=np.random.default_rng(seed)\n    def matrix(i,o):return jax.device_put((rng.normal(size=(i,o))*.5/math.sqrt(i)).astype(np.float32))\n    blocks=[]\n    for _ in range(layers):\n        blocks.append(dict(norm1=jnp.ones(width),norm2=jnp.ones(width),\n            proj=tuple(matrix(width,heads*head_dim) for _ in range(6)),out=matrix(heads*head_dim,width),\n            up=matrix(width,ff),gate=matrix(width,ff),down=matrix(ff,width)))\n    return dict(embedding=matrix(vocab,width),blocks=tuple(blocks),norm=jnp.ones(width))\n\ndef lm_loss(kernel,params,tokens,targets):\n    x=params[\'embedding\'][tokens]; B,L,W=x.shape;H=6;D=128\n    def norm(x,gain):return x*jax.lax.rsqrt(jnp.mean(x*x,-1,keepdims=True)+1e-6)*gain\n    def mm(x,w):return jnp.matmul(x,w,precision=PREC)\n    def layer(x,p):\n        z=norm(x,p[\'norm1\']);qs=[mm(z,pj).reshape(B,L,H,D) for pj in p[\'proj\']]\n        q,k,v,w,b,g=qs\n        q=q*jax.lax.rsqrt(jnp.sum(q*q,-1,keepdims=True)+1e-6)\n        k=k*jax.lax.rsqrt(jnp.sum(k*k,-1,keepdims=True)+1e-6)\n        b=jax.nn.sigmoid(b);g=-jax.nn.softplus(g-3.)\n        h0=jnp.full((B,H,D,D),.01,jnp.float32)\n        o,_=kernel(q,k,v,w,b,g,h0)\n        x=x+mm(o.reshape(B,L,H*D),p[\'out\'])\n        z=norm(x,p[\'norm2\'])\n        return x+mm(jax.nn.silu(mm(z,p[\'gate\']))*mm(z,p[\'up\']),p[\'down\'])\n    for block in params[\'blocks\']:x=jax.checkpoint(layer)(x,block)\n    x=norm(x,params[\'norm\']).reshape(-1,W)\n    y=targets.reshape(-1);N=x.shape[0];C=128\n    def ce(_,xy):\n        xx,yy=xy;logits=mm(xx,params[\'embedding\'].T)\n        val=jax.nn.logsumexp(logits,-1)-jnp.take_along_axis(logits,yy[:,None],axis=-1)[:,0]\n        return None,jnp.sum(val)\n    _,totals=jax.lax.scan(jax.checkpoint(ce),None,(x.reshape(N//C,C,W),y.reshape(N//C,C)))\n    return jnp.sum(totals)/N\n\ndef objective(kernel,micro):\n    def loss(params,tokens,targets):\n        B,L=tokens.shape\n        def one(_,xy):return None,lm_loss(kernel,params,*xy)\n        _,vals=jax.lax.scan(jax.checkpoint(one),None,\n            (tokens.reshape(B//micro,micro,L),targets.reshape(B//micro,micro,L)))\n        return jnp.mean(vals)\n    return loss\n\ndef update_for(loss):\n    def update(params,m,v,step,tokens,targets):\n        val,grads=jax.value_and_grad(loss)(params,tokens,targets)\n        step=step+1\n        m=jax.tree.map(lambda m,g:.9*m+.1*g,m,grads)\n        v=jax.tree.map(lambda v,g:.999*v+.001*g*g,v,grads)\n        new=jax.tree.map(lambda p,mm,vv:p-1e-4*(mm/(1-.9**step))/(jnp.sqrt(vv/(1-.999**step))+1e-8),params,m,v)\n        return val,new,m,v,step\n    return update\n\ndef run(a):\n    paths,cfg=make_paths(128,a.profile)\n    param=init(0);zeros=jax.tree.map(jnp.zeros_like,param)\n    rng=np.random.default_rng(0)\n    data=jax.device_put(rng.integers(0,32768,(a.batch,a.length+1),dtype=np.int32))\n    args=(param,zeros,zeros,jnp.array(0,jnp.int32),data[:,:-1],data[:,1:])\n    report=dict(kind=\'end_to_end\',protocol=\'New fixed-state replay of complete LM forward+CE+backward+Adam update\',\n        layers=5,width=384,ffn=768,vocab=32768,heads=6,head_dim=128,tied_embedding=True,\n        params=sum(x.size for x in jax.tree.leaves(param)),batch=a.batch,length=a.length,\n        microbatch=a.microbatch,profile=a.profile,config=cfg,environment=environment(),\n        tolerance_rel_l2=TOL,repeats=a.repeats,warmup=a.warmup,compile_excluded=True,\n        data_loading_excluded=True,ce_chunk_tokens=128,seeds=[],historical_200_17_reproduction=False)\n    exes={};gradf={};compile_s={}\n    for name in (\'associative_scan\',\'jax_wy\',\'pallas\'):\n        loss=objective(paths[name],a.microbatch)\n        print(\'E2E_COMPILE\',name,flush=True)\n        gradf[name],_=compile_fn(jax.value_and_grad(loss),(param,args[-2],args[-1]))\n        exes[name],compile_s[name]=compile_fn(update_for(loss),args)\n    report[\'compile_seconds\']=compile_s\n    for seed in a.seeds:\n        param=init(seed);zeros=jax.tree.map(jnp.zeros_like,param)\n        rng=np.random.default_rng(seed);host=rng.integers(0,32768,(a.batch,a.length+1),dtype=np.int32)\n        data=jax.device_put(host);args=(param,zeros,zeros,jnp.array(0,jnp.int32),data[:,:-1],data[:,1:])\n        oracle=gradf[\'associative_scan\'](param,args[-2],args[-1]);jax.block_until_ready(oracle)\n        names=[\'loss\']+[str(i) for i in range(len(jax.tree.leaves(param)))]\n        checks={name:tree_comparison(gradf[name](param,args[-2],args[-1]),oracle,names) for name in (\'jax_wy\',\'pallas\')}\n        # Compare the entire update too, including moments, not just scalar loss.\n        ref=exes[\'associative_scan\'](*args)\n        update_names=[str(i) for i in range(len(jax.tree.leaves(ref)))]\n        update_checks={name:tree_comparison(exes[name](*args),ref,update_names) for name in (\'jax_wy\',\'pallas\')}\n        row=dict(seed=seed,input_sha256=hashlib.sha256(host.tobytes()).hexdigest(),checks=checks,update_checks=update_checks)\n        row[\'passed\']=all(x[\'passed\'] for x in (*checks.values(),*update_checks.values()))\n        if row[\'passed\']:\n            from honest_gdn2 import timings\n            row[\'timing\']=timings(exes,args,a.repeats,a.warmup,seed)\n            samples=row[\'timing\'][\'samples\'];row[\'speedups\']={name:samples[name][\'median_ms\']/samples[\'pallas\'][\'median_ms\']\n                for name in (\'associative_scan\',\'jax_wy\')}\n            row[\'tokens_per_second\']={name:a.batch*a.length/v[\'median_ms\']*1000 for name,v in samples.items()}\n        report[\'seeds\'].append(row)\n        print(\'E2E_SEED\',seed,\'passed\',row[\'passed\'],\'ratios\',row.get(\'speedups\'),flush=True)\n    report[\'status\']=\'passed\' if all(x[\'passed\'] for x in report[\'seeds\']) else \'gate_failed\'\n    if report[\'status\']==\'passed\':report[\'speedups\']={name:dict(mean=statistics.mean(x[\'speedups\'][name] for x in report[\'seeds\']),\n        per_seed=[x[\'speedups\'][name] for x in report[\'seeds\']]) for name in (\'associative_scan\',\'jax_wy\')}\n    return report\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--batch\',type=int,default=8);p.add_argument(\'--length\',type=int,default=2048)\n    p.add_argument(\'--microbatch\',type=int,default=2);p.add_argument(\'--profile\',choices=(\'strict\',\'production\'),default=\'strict\')\n    p.add_argument(\'--repeats\',type=int,default=10);p.add_argument(\'--warmup\',type=int,default=3)\n    p.add_argument(\'--seeds\',nargs=\'+\',type=int,default=[0,1,2]);p.add_argument(\'--output\',default=\'honest-e2e.json\')\n    a=p.parse_args();assert a.batch%a.microbatch==0 and a.length%128==0\n    jax.config.update(\'jax_default_matmul_precision\',\'highest\')\n    assert jax.default_backend()==\'tpu\' and jax.local_device_count()==1\n    try:report=run(a)\n    except Exception as e:report=dict(status=\'error\',error=type(e).__name__,detail=str(e),traceback=traceback.format_exc(),environment=environment())\n    report[\'benchmark_sha256\']=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n    report[\'kernel_benchmark_sha256\']=hashlib.sha256(Path(__file__).with_name(\'honest_gdn2.py\').read_bytes()).hexdigest()\n    Path(a.output).write_text(json.dumps(report,indent=2,allow_nan=False))\n    print(\'FINAL_E2E\',report[\'status\'],report.get(\'speedups\'),flush=True)\n    return 0 if report[\'status\']==\'passed\' else 1\n\nif __name__==\'__main__\':raise SystemExit(main())\n',encoding='utf-8')


## run_honest_suite.py

In [ ]:
(source_dir/'benchmarks'/'run_honest_suite.py').write_text('"""Sequential fresh-process suite; saves failures and every raw timing, no publication."""\nimport hashlib,json,os,subprocess,sys,time\nfrom datetime import datetime,timezone\nfrom pathlib import Path\n\nHERE=Path(__file__).resolve().parent\nOUT=Path(os.environ.get(\'ATOMIC_BENCH_RESULTS\',\'/content/atomic-honest-results\'))\nOUT.mkdir(parents=True,exist_ok=True)\njobs=[]\nfor profile in (\'strict\',\'production\'):\n    for heads in (2,4,6):\n        for length in (2048,4096):\n            name=f\'kernel-{profile}-h{heads}-l{length}\'\n            jobs.append((name,[str(HERE/\'honest_gdn2.py\'),\'--shape\',f\'4,{length},{heads},128\',\'--profile\',profile],1800))\nfor profile in (\'strict\',\'production\'):\n    jobs.append((f\'e2e-{profile}\',[str(HERE/\'honest_e2e.py\'),\'--profile\',profile],3600))\nstate=dict(started_utc=datetime.now(timezone.utc).isoformat(),status=\'running\',cases=[])\ndef save():\n    (OUT/\'suite-status.json\').write_text(json.dumps(state,indent=2))\nsave()\nfor name,args,limit in jobs:\n    dest=OUT/(name+\'.json\');log=OUT/(name+\'.log\')\n    begin=time.time();row=dict(name=name,command=[sys.executable,*args,\'--output\',str(dest)],started=begin)\n    state[\'active\']=name;save();print(\'START\',name,flush=True)\n    try:\n        with log.open(\'w\') as output:\n            result=subprocess.run(row[\'command\'],stdout=output,stderr=subprocess.STDOUT,timeout=limit)\n        row[\'exit_code\']=result.returncode\n        if dest.exists():\n            data=json.loads(dest.read_text());row[\'status\']=data[\'status\'];row[\'speedups\']=data.get(\'speedups\')\n            row[\'result_sha256\']=hashlib.sha256(dest.read_bytes()).hexdigest()\n            row[\'detail\']=data.get(\'detail\')\n        else:row[\'status\']=\'process_failed_without_json\'\n    except subprocess.TimeoutExpired:row[\'status\']=\'timeout\'\n    except Exception as error:row.update(status=\'runner_error\',error=type(error).__name__,detail=str(error))\n    row[\'elapsed_seconds\']=time.time()-begin\n    row[\'log_sha256\']=hashlib.sha256(log.read_bytes()).hexdigest() if log.exists() else None\n    state[\'cases\'].append(row);save();print(\'FINISH\',name,row[\'status\'],row.get(\'speedups\'),flush=True)\nstate.update(status=\'completed\',active=None,finished_utc=datetime.now(timezone.utc).isoformat())\nsave()\n',encoding='utf-8')


## Execute and retain logs

In [ ]:
import os, subprocess, sys
with open('/content/atomic-suite.log','w') as log:
    suite=subprocess.Popen([sys.executable,'-u',str(source_dir/'benchmarks'/'run_honest_suite.py')],
        stdout=log,stderr=subprocess.STDOUT,env={**os.environ,'PYTHONUNBUFFERED':'1'})
print('Suite PID:',suite.pid)


In [ ]:
print('Process exit:',suite.poll())
print(pathlib.Path('/content/atomic-suite.log').read_text()[-6000:])
status=pathlib.Path('/content/atomic-honest-results/suite-status.json')
if status.exists():print(status.read_text())


In [ ]:
import shutil
shutil.make_archive('/content/atomic-honest-results','zip','/content/atomic-honest-results')
from google.colab import files
files.download('/content/atomic-honest-results.zip')
